# KNN From Scratch
## Machine learning for energy applications — Module 1, Exercise 2

In this exercise we implement the **K-Nearest Neighbours (KNN)** algorithm from scratch using only NumPy. KNN is one of the simplest and most intuitive classification algorithms: to predict the class of an unseen data point, we find the K closest points in the training set and let them vote.

By the end of this notebook you will:
- Understand the geometry behind KNN (Euclidean distance)
- Implement every component of the algorithm yourself
- Evaluate how the choice of K and data noise affect accuracy
- Verify your implementation against scikit-learn

---
### 1 — Generate Synthetic Data

We use `sklearn.datasets.make_blobs` to create a simple 2-D dataset with four clusters. This gives us a controlled environment where KNN should perform well.

We hold out 20 % of the points as **validation** rows. In this notebook they are used to compare values of K and to check the implementation against scikit-learn; we make no final performance claim about this clean dataset, so it needs no separate test set. Section 8, where we do choose a K, adds one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generate synthetic data: 200 samples, 2 features, 4 clusters
np.random.seed(42)
X, y = make_blobs(n_samples=200, n_features=2, centers=4,
                  cluster_std=1.0, random_state=42)

# Training / validation split (80/20) — manual to avoid importing sklearn.model_selection
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_val = X[indices[:split]], X[indices[split:]]
y_train, y_val = y[indices[:split]], y[indices[split:]]

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                     cmap='viridis', edgecolors='k', s=50, alpha=0.8,
                     label='Training')
ax.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap='viridis',
           edgecolors='r', s=80, marker='D', alpha=0.9, label='Validation')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Synthetic Data — 4 Clusters')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Class')
plt.tight_layout()
plt.show()

---
### 2 — Euclidean Distance

The Euclidean distance between two vectors $\mathbf{a}$ and $\mathbf{b}$ in $\mathbb{R}^d$ is:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{d}(a_i - b_i)^2}$$

Implement this using NumPy (no loops needed).

In [ ]:
def euclidean_distance(a, b):
    """Compute the Euclidean distance between two vectors.

    Parameters
    ----------
    a : np.ndarray, shape (d,)
    b : np.ndarray, shape (d,)

    Returns
    -------
    float
    """
    return np.sqrt(np.sum((a - b) ** 2))


# Quick sanity check
p1 = np.array([0.0, 0.0])
p2 = np.array([3.0, 4.0])
print(f"Distance between {p1} and {p2}: {euclidean_distance(p1, p2):.4f}")
assert np.isclose(euclidean_distance(p1, p2), 5.0), "Expected 5.0"
print("Sanity check passed.")

---
### 3 — Find K Nearest Neighbours

Given a query point, compute its distance to every training sample and return the indices of the K closest ones.

In [ ]:
def get_neighbors(X_train, query_point, k):
    """Return the indices of the k nearest neighbours in X_train.

    Parameters
    ----------
    X_train : np.ndarray, shape (n, d)
    query_point : np.ndarray, shape (d,)
    k : int

    Returns
    -------
    np.ndarray of int, shape (k,)
        Indices into X_train of the k nearest neighbours.
    """
    distances = np.array([euclidean_distance(query_point, x) for x in X_train])
    neighbor_indices = np.argsort(distances)[:k]
    return neighbor_indices


# Quick test: find 3 nearest neighbours for the first validation point
query = X_val[0]
neighbor_idx = get_neighbors(X_train, query, k=3)
print(f"Query point: {query}")
print(f"Nearest neighbour indices: {neighbor_idx}")
print(f"Neighbour labels: {y_train[neighbor_idx]}")

---
### 4 — Majority Vote

Given the labels of the K nearest neighbours, return the most common label. In the event of a tie, the label that appears first among the neighbours wins.

In [ ]:
def majority_vote(neighbor_labels):
    """Return the most common label (majority vote).

    Parameters
    ----------
    neighbor_labels : np.ndarray of int, shape (k,)

    Returns
    -------
    int
        Predicted class label.
    """
    labels, counts = np.unique(neighbor_labels, return_counts=True)
    max_count = np.max(counts)
    # If there is a tie, pick the label that appears first among neighbours
    candidates = labels[counts == max_count]
    for label in neighbor_labels:
        if label in candidates:
            return int(label)


# Quick test
print(f"Majority of [0, 0, 1]: {majority_vote(np.array([0, 0, 1]))}")
print(f"Majority of [1, 2, 2]: {majority_vote(np.array([1, 2, 2]))}")
assert majority_vote(np.array([0, 0, 1])) == 0
assert majority_vote(np.array([1, 2, 2])) == 2
print("Majority vote tests passed.")

---
### 5 — Predict Function

Combine the building blocks above into a single `predict` function that classifies every point in a set of query points.

In [ ]:
def predict(X_train, y_train, X_query, k):
    """Predict class labels for every row in X_query using KNN.

    Parameters
    ----------
    X_train : np.ndarray, shape (n_train, d)
    y_train : np.ndarray, shape (n_train,)
    X_query : np.ndarray, shape (n_query, d)
    k       : int

    Returns
    -------
    np.ndarray of int, shape (n_query,)
    """
    predictions = np.empty(len(X_query), dtype=int)
    for i, query in enumerate(X_query):
        neighbor_idx = get_neighbors(X_train, query, k)
        neighbor_labels = y_train[neighbor_idx]
        predictions[i] = majority_vote(neighbor_labels)
    return predictions


# Quick test on the validation rows
y_pred = predict(X_train, y_train, X_val, k=3)
print(f"First 10 predictions: {y_pred[:10]}")
print(f"First 10 true labels: {y_val[:10]}")

---
### 6 — Accuracy Function

Implement a simple accuracy metric: the fraction of predictions that match the true labels.

In [ ]:
def accuracy(y_true, y_pred):
    """Compute classification accuracy.

    Parameters
    ----------
    y_true : np.ndarray
    y_pred : np.ndarray

    Returns
    -------
    float
        Accuracy in [0, 1].
    """
    return np.mean(y_true == y_pred)


# Quick test
assert accuracy(np.array([1, 2, 3]), np.array([1, 2, 3])) == 1.0
assert accuracy(np.array([1, 2, 3]), np.array([1, 0, 0])) == 1 / 3
print("Accuracy function tests passed.")

---
### 7 — Evaluate on Clean Data (K = 3, 5, 7)

Evaluate our KNN implementation on the well-separated clusters generated earlier. We sweep over several values of K and plot the validation accuracy for each.

In [ ]:
k_values = [3, 5, 7]
clean_accuracies = []

print("Clean data, validation accuracy (cluster_std=1.0)")
print("-" * 35)
for k in k_values:
    y_pred = predict(X_train, y_train, X_val, k)
    acc = accuracy(y_val, y_pred)
    clean_accuracies.append(acc)
    print(f"  K = {k}  |  Accuracy = {acc:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([str(k) for k in k_values], clean_accuracies, color='steelblue',
              edgecolor='k')
ax.set_xlabel('K')
ax.set_ylabel('Validation accuracy')
ax.set_title('KNN Accuracy on Clean Data (cluster_std=1.0)')
ax.set_ylim(0, 1.05)
for bar, acc in zip(bars, clean_accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{acc:.2f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

---
### 8 — Evaluate on Noisy Data, Then Test the Chosen K Once

Now we regenerate the data with `cluster_std=3.0` so that neighbouring clusters overlap. This makes classification harder and shows how the choice of K affects performance: with overlapping classes, a single neighbour (K = 1) follows the noise.

Here we *choose* K, so we split the points three ways: 60 % **training**, 20 % **validation** and 20 % **test**. We compare K on the validation rows only and lock the K with the highest validation accuracy (the smallest such K on a tie). That best validation score is optimistic, because we picked it as the maximum of eight noisy numbers, so we then score the locked K once on the test rows and report that number.

In [ ]:
# Generate noisy data
np.random.seed(42)
X_noisy, y_noisy = make_blobs(n_samples=200, n_features=2, centers=4,
                              cluster_std=3.0, random_state=42)

# Training / validation / test split (60/20/20): we will choose K, so we need test rows
indices_noisy = np.random.permutation(len(X_noisy))
n_tr, n_va = int(0.6 * len(X_noisy)), int(0.2 * len(X_noisy))
tr_idx, va_idx, te_idx = np.split(indices_noisy, [n_tr, n_tr + n_va])
X_train_n, X_val_n, X_test_n = X_noisy[tr_idx], X_noisy[va_idx], X_noisy[te_idx]
y_train_n, y_val_n, y_test_n = y_noisy[tr_idx], y_noisy[va_idx], y_noisy[te_idx]
print(f"Training: {len(tr_idx)}  validation: {len(va_idx)}  test (closed): {len(te_idx)}")

# Sweep a wider range of K values on the validation rows
k_range = [1, 3, 5, 7, 9, 11, 15, 21]
noisy_accuracies = []

print("Noisy data, validation accuracy (cluster_std=3.0)")
print("-" * 35)
for k in k_range:
    y_pred_n = predict(X_train_n, y_train_n, X_val_n, k)
    acc = accuracy(y_val_n, y_pred_n)
    noisy_accuracies.append(acc)
    print(f"  K = {k:2d}  |  Accuracy = {acc:.4f}")

# Visualize the noisy data + accuracy curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter plot
ax = axes[0]
ax.scatter(X_train_n[:, 0], X_train_n[:, 1], c=y_train_n, cmap='viridis',
           edgecolors='k', s=50, alpha=0.7)
ax.scatter(X_val_n[:, 0], X_val_n[:, 1], c=y_val_n, cmap='viridis',
           edgecolors='r', s=80, marker='D', alpha=0.9)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Noisy Data — cluster_std=3.0 (diamonds: validation rows)')

# Right: accuracy vs K
ax = axes[1]
ax.plot(k_range, noisy_accuracies, 'o-', color='coral', linewidth=2,
        markersize=8)
ax.set_xlabel('K')
ax.set_ylabel('Validation accuracy')
ax.set_title('Validation Accuracy vs K on Noisy Data')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Lock K on validation accuracy (np.argmax returns the first, i.e. smallest, K on a tie),
# then open the test rows once
best_k = k_range[int(np.argmax(noisy_accuracies))]
test_acc = accuracy(y_test_n, predict(X_train_n, y_train_n, X_test_n, best_k))
print(f"Locked K = {best_k} (validation accuracy {max(noisy_accuracies):.4f})")
print(f"FINAL TEST (used once): accuracy {test_acc:.4f} on {len(te_idx)} test rows")

---
### 9 — Compare with scikit-learn KNeighborsClassifier

To validate our scratch implementation, we compare against the battle-tested `sklearn.neighbors.KNeighborsClassifier` on the validation rows of the clean data. The accuracies should match. (This checks the code, not the model, so reusing the validation rows is fine.)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print(f"{'K':>3}  {'Ours':>10}  {'sklearn':>10}  {'Match':>6}")
print("-" * 38)

for k in [3, 5, 7]:
    # Our implementation — clean data, validation rows
    our_pred = predict(X_train, y_train, X_val, k)
    our_acc = accuracy(y_val, our_pred)

    # sklearn — clean data, validation rows
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    sk_acc = clf.score(X_val, y_val)

    match = np.isclose(our_acc, sk_acc)
    print(f"  {k}    {our_acc:.4f}      {sk_acc:.4f}    {'Yes' if match else 'No'}")

print("\nIf all rows show 'Yes', the scratch implementation agrees with sklearn.")

---
### Summary

In this exercise we:

1. Generated and visualized synthetic 2-D cluster data, with training and validation rows.
2. Implemented the Euclidean distance function from scratch.
3. Built `get_neighbors` to find the K closest training samples.
4. Built `majority_vote` to turn neighbour labels into a prediction.
5. Combined everything into a `predict` function.
6. Wrote a simple `accuracy` evaluator.
7. Observed near-perfect validation accuracy on clean, well-separated data.
8. Showed that accuracy degrades on noisy data and that the choice of K matters; chose K on validation rows and scored it once on test rows.
9. Verified our implementation matches scikit-learn.

**Key takeaway:** KNN is conceptually simple — no training phase, just geometry. But its performance depends heavily on data quality and the hyperparameter K. In real energy-industry applications (e.g., lithology classification from well logs), feature scaling (fitted on training rows only), thoughtful K selection on validation data, and splits by well are essential.